# Workshop Tutorial: Serving DeepSeek on AMD GPUs with **SGLang** (+ PD Disaggregation)

Welcome to this hands-on workshop! We'll use **AMD Instinct GPUs** to serve powerful DeepSeek models with [**SGLang**](https://github.com/sgl-project/sglang), a high-performance serving engine, and then take it up a notch with the production-grade trick AMD actually uses at scale: **Prefill–Decode (PD) disaggregation**.

You'll go from *"one big kitchen"* (a single monolithic server) to a *"professional kitchen brigade"* (prefill and decode running as separate, specialized services) — and along the way build three fun applications:

1. **Advanced Chatbot** — Open WebUI with web search and file interaction.
2. **Code Development Assistant** — CodeGPT / VS Code AI Toolkit for pair programming.
3. **Web-Based AI Agent** — an autonomous browsing agent wired to your endpoint.

Everything talks to an **OpenAI-compatible API**, so the apps don't care whether they're hitting one server or a whole disaggregated cluster.

Let's dive in!

## Table of Contents

- [Step 1: Access AMD GPUs](#step1)
- [Step 2: Docker Installation and Configuration](#step2)
- [Step 3: One Big Kitchen — Launch a Single SGLang Server](#step3)
- [Step 4: The Kitchen Brigade — PD Disaggregation 🍳](#step4)
- [Step 5: Advanced Chatbot with Open WebUI](#step5)
- [Step 6: Chatbot Testing with DeepSeek](#step6)
- [Step 7: Code Development Assistant (CodeGPT / AI Toolkit)](#step7)
- [Step 8: Build a Snake Game](#step8)
- [Step 9: Optional Challenge — Pac-Man](#step9)
- [Step 10: Deploy a Web-Based AI Agent](#step10)

<a id="step1"></a>

## Step 1: Access AMD GPUs

- **For Developers:** Request access via AMD's Developer Cloud application [here](https://www.amd.com/en/forms/registration/developer-cloud-application.html).
- **Enterprise Users:** Check out AMD's cloud partners [here](https://www.amd.com/en/developer/resources/rocm-hub/dev-ai.html).

> 💡 For the **baseline** (Step 3) a single AMD Instinct node is enough. For the **PD disaggregation** finale (Step 4) you'll want **two nodes** (one prefill, one decode) connected by RDMA — exactly the kind of setup the AMD cloud provides.

<a id="step2"></a>

## Step 2: Docker Installation and Configuration

Ensure Docker is installed and configured. Follow [Docker's official guide](https://docs.docker.com/get-docker/) if needed.

**Allow non-root Docker access:**

```bash
sudo usermod -aG docker $USER
newgrp docker
```

**Verify Docker:**

```bash
docker run hello-world
```

<a id="step3"></a>

## Step 3: One Big Kitchen 🍲 — Launch a Single SGLang Server

Before we get fancy, let's run the simplest possible setup: **one container, one server, doing everything**. Think of it as a single chef who reads the whole order *and* plates every dish. Perfect for getting the chatbot and code apps working quickly.

### Start the SGLang container

```bash
docker run -it --rm \
  --network=host \
  --device=/dev/kfd \
  --device=/dev/dri \
  --group-add=video \
  --ipc=host \
  --cap-add=SYS_PTRACE \
  --security-opt seccomp=unconfined \
  --shm-size 16G \
  -v /home/amd/models:/models \
  lmsysorg/sglang-rocm:v0.5.13.post1-rocm720-mi35x-20260618
```

> 🐳 **Image note:** `lmsysorg/sglang-rocm:v0.5.13.post1-rocm720-mi35x-20260618` is the ROCm build used in this workshop. Check [Docker Hub](https://hub.docker.com/r/lmsysorg/sglang-rocm/tags) for the latest `-rocm` tag that matches your GPU/ROCm version.

### Launch the server (OpenAI-compatible)

Run **inside** the container:

```bash
SGLANG_USE_AITER=1 \
python3 -m sglang.launch_server \
    --model-path /models/DeepSeek-V4-Pro \
    --host 0.0.0.0 \
    --port 30000 \
    --tp-size 8 \
    --trust-remote-code
```

> 🧠 `DeepSeek-V4-Pro` is a large Mixture-of-Experts model — `--tp-size 8` spreads it across all 8 GPUs of one node. This single server does **both** prefill and decode; in Step 4 we split those apart.

A few things to notice (this is *not* vLLM anymore 🙂):

| Concept | vLLM | **SGLang** |
|---|---|---|
| Module | `vllm serve <model>` | `python3 -m sglang.launch_server` |
| Model flag | positional | `--model-path` |
| Default port | 8000 | **30000** |
| Tensor parallel | `--tensor-parallel-size` | `--tp-size` |

> 🌐 **Public access:** Pick a port open to the public and confirm firewall rules. Get your server's public IP with `curl ifconfig.me`.

### Verify the endpoint

From your laptop (or the next code cell):

```bash
curl http://YOUR_SERVER_PUBLIC_IP:30000/v1/models
```

### 🔎 Optional: verify + measure first-token latency from Python

The cell below hits `/v1/models` and then streams a completion, printing the **TTFT** (time to first token) — the metric PD disaggregation is designed to improve. Edit `BASE_URL` to point at your server.

In [ ]:
# pip install openai httpx  (already present in most SGLang images)
import time, httpx
from openai import OpenAI

BASE_URL = "http://localhost:30000/v1"   # <-- your SGLang endpoint (or the PD router from Step 4)
API_KEY  = "EMPTY"                        # SGLang needs no key unless you pass --api-key

# 1) What model is being served?
models = httpx.get(f"{BASE_URL}/models", timeout=10).json()
served = models["data"][0]["id"]
print("Served model:", served)

# 2) Stream a prompt and time the first token
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
t0 = time.time()
ttft = None
stream = client.chat.completions.create(
    model=served,
    messages=[{"role": "user", "content": "In one sentence, why separate prefill from decode?"}],
    stream=True,
    max_tokens=128,
)
text = []
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    if delta and ttft is None:
        ttft = time.time() - t0
    text.append(delta)
print(f"\nTTFT: {ttft*1000:.0f} ms")
print("Answer:", "".join(text).strip())


<a id="step4"></a>

## Step 4: The Kitchen Brigade 🍳 — PD Disaggregation

One chef doing *everything* is fine for a snack. But a real restaurant runs a **brigade**: stations that each do one thing extremely well. LLM serving has exactly two "stations":

| Phase | Restaurant role | What it does | Bottleneck |
|---|---|---|---|
| **Prefill** | 🔪 **Prep station** (*mise en place*) | Reads the **entire prompt** at once, builds the **KV cache** | Compute-bound |
| **Decode** | 🍽️ **Plating line** | Emits tokens **one at a time** using the KV cache | Memory-bound |

In a single server these two fight over the same kitchen: a big incoming **prefill** order *interrupts* the steady stream of **decode** plates (hurting per-token latency), and with data-parallel attention one worker may be prepping while another is plating — an imbalance that stalls everyone.

**PD disaggregation** gives each station its own kitchen. The prompt is prepped on a **prefill server**, the finished **KV-cache "tray" is shipped over RDMA** to a **decode server**, which plates the tokens. A **router** plays *expediter*, sending each order to the right stations.

```
                         ┌──────────────────────────────┐
   client  ──request──▶  │   ROUTER (expediter)         │   sglang_router
                         │   :30000  /v1/chat/...        │   --pd-disaggregation
                         └───────┬───────────────┬──────┘
                                 │ 1. "prep this" │ 3. "plate this"
                                 ▼                ▼
                  ┌───────────────────┐   ┌───────────────────┐
                  │ 🔪 PREFILL server │   │ 🍽️ DECODE server  │
                  │  --disaggregation │   │  --disaggregation │
                  │     -mode prefill │   │     -mode decode  │
                  │  builds KV cache  │   │  generates tokens │
                  └─────────┬─────────┘   └───────────────────┘
                            │  2. KV-cache tray over RDMA
                            └──────────(mooncake + MORI)──────▶
```

> 📚 References: [SGLang PD docs](https://docs.sglang.ai/advanced_features/pd_disaggregation.html) · [AMD ROCm: Disaggregating Prefill & Decode with SGLang on Instinct](https://rocm.blogs.amd.com/software-tools-optimization/disaggregation/README.html)

### 🗝️ Decoder ring — the flags that matter

The real commands below carry a *wall* of tuning env-vars. Don't panic — here are the only ones you need to understand to follow the architecture:

| Flag | Meaning |
|---|---|
| `--disaggregation-mode prefill\|decode` | Which **station** this server is. The one knob that turns a normal server into a brigade member. |
| `--disaggregation-transfer-backend mooncake` | The **delivery service** that ships KV-cache trays between stations. |
| `--moe-a2a-backend mori` | **MORI** all-to-all backend for the Mixture-of-Experts dispatch (AMD's high-perf path). |
| `--disaggregation-ib-device ionic_0,...` | The **RDMA NICs** (here `ionic_*`) used to move trays fast between nodes. |
| `--tp-size / --ep-size / --dp-size` | Tensor / Expert / Data parallel widths across the 8 GPUs. |

Everything prefixed `SGLANG_*` / `MORI_*` / `NCCL_*` is performance tuning for this specific MI35x cluster — copy it as-is.

### 🔪 Prefill server (the prep station)

Run on the **prefill node**. Pull the image first:

```bash
docker pull lmsysorg/sglang-rocm:v0.5.13.post1-rocm720-mi35x-20260618
```

📋 **Full command (copy-paste):**

```bash
SGLANG_USE_AITER=1                      \
SGLANG_MORI_DISPATCH_DTYPE=auto         \
SGLANG_MORI_COMBINE_DTYPE=auto          \
SGLANG_MORI_QP_PER_TRANSFER=4           \
SGLANG_MORI_NUM_WORKERS=4               \
MORI_IO_SQ_BACKOFF_TIMEOUT_US=50000     \
MORI_IO_QP_MAX_SEND_WR=16384            \
MORI_IO_QP_MAX_CQE=32768                \
MORI_IO_QP_MAX_SGE=4                    \
SGLANG_DISAGGREGATION_BOOTSTRAP_TIMEOUT=3600  \
SGLANG_DISAGGREGATION_WAITING_TIMEOUT=3600    \
MORI_SHMEM_MODE=ISOLATION                     \
SGLANG_ENABLE_SPEC_V2=1                        \
SGLANG_ENABLE_OVERLAP_PLAN_STREAM=1            \
SGLANG_LOG_MS=true                             \
SGLANG_DISAGGREGATION_NUM_PRE_ALLOCATE_REQS=32 \
MORI_MAX_DISPATCH_TOKENS_PREFILL=8192           \
MORI_MAX_DISPATCH_TOKENS_DECODE=256             \
MORI_MOE_MAX_INPUT_TOKENS_DECODE=2048           \
SGLANG_MORI_DISPATCH_INTER_KERNEL_SWITCH_THRESHOLD=4096 \
MORI_EP_LAUNCH_CONFIG_MODE=AUTO                         \
MORI_APP_LOG_LEVEL=INFO  \
MORI_RDMA_SL=3   \
MORI_RDMA_TC=96  \
MORI_IO_SL=3     \
MORI_IO_TC=96    \
MORI_IO_TC_DISABLE=0  \
NCCL_IB_HCA=ionic_0,ionic_1,ionic_2,ionic_3,ionic_4,ionic_5,ionic_6,ionic_7 \
GLOO_SOCKET_IFNAME=enp81s0f1 \
NCCL_SOCKET_IFNAME=enp81s0f1 \
SGLANG_MORI_NUM_MAX_DISPATCH_TOKENS_PER_RANK=16384 \
SGLANG_DEFAULT_THINKING=1                       \
SGLANG_DSV4_REASONING_EFFORT=max                \
SGLANG_OPT_DEEPGEMM_HC_PRENORM=false            \
SGLANG_USE_ROCM700A=0                           \
SGLANG_OPT_USE_FUSED_COMPRESS=true              \
SGLANG_HACK_FLASHMLA_BACKEND=unified_kv_triton  \
SGLANG_OPT_FP8_WO_A_GEMM=false                  \
SGLANG_OPT_USE_JIT_INDEXER_METADATA=false       \
SGLANG_OPT_USE_TOPK_V2=false                    \
SGLANG_OPT_USE_AITER_INDEXER=true               \
SGLANG_OPT_USE_TILELANG_INDEXER=false           \
SGLANG_OPT_USE_TILELANG_MHC_PRE=false           \
SGLANG_OPT_USE_TILELANG_MHC_POST=false          \
SGLANG_FP8_PAGED_MQA_LOGITS_TORCH=1             \
SGLANG_OPT_USE_FUSED_COMPRESS_TRITON=true       \
SGLANG_OPT_USE_MULTI_STREAM_OVERLAP=false       \
SGLANG_ROCM_USE_MULTI_STREAM=false              \
AITER_BF16_FP8_MOE_BOUND=0                       \
SGLANG_EAGER_INPUT_NO_COPY=true                 \
python3 -m sglang.launch_server \
    --model-path /models/DeepSeek-V4-Pro \
    --disaggregation-mode prefill \
    --disaggregation-ib-device ionic_0,ionic_1,ionic_2,ionic_3,ionic_4,ionic_5,ionic_6,ionic_7 \
    --host 0.0.0.0 --port 8000 --trust-remote-code \
    --tp-size 8 --ep-size 8 --dp-size 8 --decode-log-interval 100 --watchdog-timeout 3600 \
    --load-balance-method round_robin --kv-cache-dtype fp8_e4m3 \
    --attention-backend dsv4 \
    --page-size 256 \
    --swa-full-tokens-ratio 0.1 \
    --disable-shared-experts-fusion \
    --tool-call-parser deepseekv4 \
    --reasoning-parser deepseek-v4 \
    --disaggregation-transfer-backend mooncake --moe-a2a-backend mori --deepep-mode normal \
    --enable-dp-attention --moe-dense-tp-size 1 --enable-dp-lm-head \
    --mem-fraction-static 0.8 \
    --chunked-prefill-size 131072 \
    --context-length 9217 \
    --max-running-requests 1024 \
    --max-total-tokens 262144 \
    --disable-cuda-graph --disable-radix-cache 2>&1 | tee x0829p
```

The prep station signature: **`--disaggregation-mode prefill`**, a large `--chunked-prefill-size` (it ingests big prompts), and `--disable-radix-cache` (no plating means no need to cache decode prefixes).

### 🍽️ Decode server (the plating line)

Run on the **decode node**:

📋 **Full command (copy-paste):**

```bash
SGLANG_USE_AITER=1                      \
SGLANG_MORI_DISPATCH_DTYPE=auto         \
SGLANG_MORI_COMBINE_DTYPE=auto          \
SGLANG_MORI_QP_PER_TRANSFER=4           \
SGLANG_MORI_NUM_WORKERS=4               \
MORI_IO_SQ_BACKOFF_TIMEOUT_US=50000     \
MORI_IO_QP_MAX_SEND_WR=16384            \
MORI_IO_QP_MAX_CQE=32768                \
MORI_IO_QP_MAX_SGE=4                    \
SGLANG_DISAGGREGATION_BOOTSTRAP_TIMEOUT=3600  \
SGLANG_DISAGGREGATION_WAITING_TIMEOUT=3600    \
MORI_SHMEM_MODE=ISOLATION                     \
SGLANG_ENABLE_SPEC_V2=1                        \
SGLANG_ENABLE_OVERLAP_PLAN_STREAM=1            \
SGLANG_LOG_MS=true                             \
SGLANG_DISAGGREGATION_NUM_PRE_ALLOCATE_REQS=32 \
MORI_MAX_DISPATCH_TOKENS_DECODE=64             \
MORI_MOE_MAX_INPUT_TOKENS_DECODE=332           \
SGLANG_MORI_DISPATCH_INTER_KERNEL_SWITCH_THRESHOLD=4096 \
MORI_EP_LAUNCH_CONFIG_MODE=AUTO                         \
MORI_APP_LOG_LEVEL=INFO  \
MORI_RDMA_SL=3   \
MORI_RDMA_TC=96  \
MORI_IO_SL=3     \
MORI_IO_TC=96    \
MORI_IO_TC_DISABLE=0  \
NCCL_IB_HCA=ionic_0,ionic_1,ionic_2,ionic_3,ionic_4,ionic_5,ionic_6,ionic_7 \
GLOO_SOCKET_IFNAME=enp81s0f1 \
NCCL_SOCKET_IFNAME=enp81s0f1 \
SGLANG_MORI_NUM_MAX_DISPATCH_TOKENS_PER_RANK=128 \
SGLANG_DEFAULT_THINKING=1                       \
SGLANG_DSV4_REASONING_EFFORT=max                \
SGLANG_OPT_DEEPGEMM_HC_PRENORM=false            \
SGLANG_USE_ROCM700A=0                           \
SGLANG_OPT_USE_FUSED_COMPRESS=true              \
SGLANG_HACK_FLASHMLA_BACKEND=unified_kv_triton  \
SGLANG_OPT_FP8_WO_A_GEMM=false                  \
SGLANG_OPT_USE_JIT_INDEXER_METADATA=false       \
SGLANG_OPT_USE_TOPK_V2=false                    \
SGLANG_OPT_USE_AITER_INDEXER=true               \
SGLANG_OPT_USE_TILELANG_INDEXER=false           \
SGLANG_OPT_USE_TILELANG_MHC_PRE=false           \
SGLANG_OPT_USE_TILELANG_MHC_POST=false          \
SGLANG_FP8_PAGED_MQA_LOGITS_TORCH=1             \
SGLANG_OPT_USE_FUSED_COMPRESS_TRITON=true       \
SGLANG_OPT_USE_MULTI_STREAM_OVERLAP=false       \
SGLANG_ROCM_USE_MULTI_STREAM=false              \
AITER_BF16_FP8_MOE_BOUND=0                       \
SGLANG_EAGER_INPUT_NO_COPY=true                 \
python3 -m sglang.launch_server \
    --model-path /models/DeepSeek-V4-Pro \
    --disaggregation-mode decode \
    --disaggregation-ib-device ionic_0,ionic_1,ionic_2,ionic_3,ionic_4,ionic_5,ionic_6,ionic_7 \
    --host 0.0.0.0 --port 8000 --trust-remote-code \
    --tp-size 8 --ep-size 8 --dp-size 8 \
    --decode-log-interval 100 --watchdog-timeout 3600 \
    --load-balance-method round_robin \
    --kv-cache-dtype fp8_e4m3 \
    --attention-backend dsv4 \
    --page-size 256 \
    --swa-full-tokens-ratio 0.1 \
    --disable-shared-experts-fusion \
    --tool-call-parser deepseekv4 \
    --reasoning-parser deepseek-v4 \
    --disaggregation-transfer-backend mooncake --moe-a2a-backend mori \
    --deepep-mode normal --enable-dp-attention \
    --moe-dense-tp-size 1 --enable-dp-lm-head \
    --mem-fraction-static 0.85 \
    --max-running-requests 1024 \
    --cuda-graph-bs $(seq 1 128 | tr '\n' ' ') \
    --prefill-round-robin-balance 2>&1 | tee x0833d
```

The plating-line signature: **`--disaggregation-mode decode`**, CUDA graphs **enabled** across many batch sizes (`--cuda-graph-bs`) for fast steady-state token emission, and smaller per-step dispatch tokens (`MORI_MAX_DISPATCH_TOKENS_DECODE=64`) — it plates little and often.

> ⚠️ Both servers use `--port 8000` **on their own nodes** — that's fine, they live on different hosts. The router (next) is the single public door.

### 🧑‍🍳 The router (the expediter)

The router exposes **one** OpenAI-compatible endpoint on `:30000` and fans requests out to the prefill + decode stations. Point all your apps here.

```bash
python -m sglang_router.launch_router \
    --pd-disaggregation \
    --port 30000 \
    --policy random \
    --prefill-policy random \
    --decode-policy random \
    --prefill http://10.235.192.82:8000 \
    --decode  http://10.235.192.88:8000
```

**Example node map** (one prefill ↔ one decode per pairing — "1P1D"):

| Pairing | Prefill node | Decode node |
|---|---|---|
| 08-29 ↔ 08-33 | `10.235.192.82:8000` | `10.235.192.88:8000` |
| 08-21 ↔ 08-25 | `10.235.192.81:8000` | `10.235.192.85:8000` |
| 09-29 ↔ 09-33 | `10.235.192.84:8000` | `10.235.192.86:8000` |
| 09-21 ↔ 09-25 | `10.235.192.87:8000` | `10.235.192.83:8000` |
| 06-21 | `10.235.192.132:8000` | — |

Scale out by passing **multiple** `--prefill`/`--decode` flags (e.g. several decode plating lines behind one prep station) and swapping `--policy random` for `cache_aware` / `power_of_two`. See the [sglang-router docs](https://docs.sglang.ai/advanced_features/router.html).

✅ **Test it:**

```bash
curl http://<ROUTER_IP>:30000/v1/models
```

From here on, **every app uses `http://<ROUTER_IP>:30000/v1`** — exactly the same interface as the single server in Step 3. The brigade is invisible to the diner. 🎉

### 🏁 Optional: a tale of two kitchens

Run the **Step 3 verify cell** twice — once with `BASE_URL` pointed at the single server, once at the PD router — and compare the **TTFT** and the tokens/sec. With realistic concurrency, the brigade keeps decode plating smooth because incoming prompts no longer interrupt the line. That's the whole point of PD disaggregation: **better, more predictable latency under load.**

<a id="step5"></a>

## Step 5: Advanced Chatbot with Open WebUI

Follow the install instructions from the [Open WebUI repo](https://github.com/open-webui/open-webui), then connect it to your SGLang endpoint:

- Open `Settings` → `Connections`.

![OpenWebUI Setup 1](assets/openwebui1.png)

- Add a connection:
  - **URL:** `http://YOUR_ENDPOINT:30000/v1` — this is either your **single server** (Step 3) or your **PD router** (Step 4). Same format either way.
  - **API Key:** whatever you set (SGLang needs none unless you passed `--api-key`; use any non-empty string if the UI requires one).
  - **Model name:** the exact `id` returned by `GET /v1/models` (e.g. `/models/DeepSeek-V4-Pro`).
  - Click `+`, then `Save`.

![OpenWebUI Setup 2](assets/openwebui2.png)

<a id="step6"></a>

## Step 6: Chatbot Testing with DeepSeek

Use Open WebUI to chat. Example prompt:

```
Imagine facing east. Turn 90° left, then 270° right, then 180° left. Which direction are you facing?
```

Follow up with a visualization request:

```
Can you give me a simple python code without importing external libraries to visualize this step-by-step with Unicode arrows?
```

![OpenWebUI Example](assets/webui_example.gif)

<a id="step7"></a>

## Step 7: Code Development Assistant (CodeGPT / VS Code AI Toolkit)

### Option A — CodeGPT (may be restricted in some countries)

Install **CodeGPT** in VS Code (Extensions → search *CodeGPT*), create an account, then:

![CodeGPT Setup 1](assets/codegpt_setup1.png)

- Open the **LLMs Cloud Model** tab.
- **Select Provider:** `Custom`.
- **Select model:** the exact `id` from `GET /v1/models` (e.g. `/models/DeepSeek-V4-Pro`).
- **Connect to Custom:**
  - **API Key:** your key (or any non-empty string).
  - **Custom Link:** `http://YOUR_ENDPOINT:30000/v1/chat/completions`

![CodeGPT Setup 2](assets/codegpt_setup2.png)

### Option B — VS Code AI Toolkit

Install **AI Toolkit for VS Code**, then click `remote inference`:

![AI Toolkit Setup 1](assets/aitoolkit1.png)

- `Add a custom model`

![AI Toolkit Setup 2](assets/aitoolkit2.png)

- **OpenAI-compatible URL:** `http://YOUR_ENDPOINT:30000/v1/chat/completions`

![AI Toolkit Setup 3](assets/aitoolkit_url.png)

- **Model name:** the exact `id` from `GET /v1/models`.

![AI Toolkit Setup 4](assets/aitoolkit3.png)

- Press enter to keep the name as-is.

![AI Toolkit Setup 5](assets/aitoolkit5.png)

- **Auth header** (only if you launched with `--api-key`): `Authorization: Bearer YOUR_API_KEY`.

![AI Toolkit Setup 6](assets/aitoolkit6.png)

Your model now appears under `My MODELS`. Click it to open the playground.

![AI Toolkit Setup 7](assets/aitoolkit.png)

<a id="step8"></a>

## Step 8: Build a Snake Game

In VS Code, ask your assistant:

```
Can you build a classic snake game? Include 'Powered by DeepSeek on AMD Instinct' in the corner. Use Python.
```

Run the generated code and enjoy!

![Generated Snake Game](assets/snake.gif)

<a id="step9"></a>

## Step 9: Optional Challenge — Pac-Man

Build a Pac-Man game in **at most three prompts**. Here's one made with a single prompt (full DeepSeek-R1 671B):

![Generated Pac-Man](assets/pacman.gif)

<a id="step10"></a>

## Step 10: Deploy a Web-Based AI Agent

Clone the customized fork: [web-ui-sglang-vllm-deepseekr-fork](https://github.com/Mahdi-CV/web-ui-sglang-vllm-deepseekr-fork).

Once the agent UI is running, connect it to your SGLang endpoint:

- In `Agent Settings`, **uncheck `Use Vision`** (this model is text-only).

![Agent Setup 1](assets/agent1.png)

- In `LLM Configuration`:
  - **LLM Provider:** `openai`
  - **Model name:** the exact `id` from `GET /v1/models`.
  - **URL:** `http://YOUR_ENDPOINT:30000/v1` (single server or PD router).
  - **API Key:** your key (or any non-empty string).

![Agent Setup 2](assets/agent2.png)

- In `Browser Settings`, enable **Headless Mode**.

![Agent Setup 3](assets/agent3.png)

- Go to `Run Agent` and give it a task.

![Agent Setup 4](assets/agent4.png)

**Example 1:** Find the trending Hugging Face text-to-image model and generate a sample image:

![Web Agent Hugging Face Example](assets/hf_webagent.gif)

**Example 2:** Gather recipe ingredients and add them to a shopping cart:

![Web Agent Dinner Recipe Shopping Example](assets/dinner_webagent.gif)

---

🎉 **You did it!** You served DeepSeek on AMD GPUs with SGLang, scaled from *one big kitchen* to a *disaggregated kitchen brigade*, and wired it into a chatbot, a coding assistant, and a web agent.

Happy serving! If you hit issues or have questions, ask during the workshop!